# 基于MindSpore的GPT模型实现情感分类

## 案例介绍

基于GPT模型，通过对imdb电影评论数据集进行微调，实现高精度的情感倾向分类（正面/负面）。

## 模型简介

GPT（Generative Pre-trained Transformer）是由 OpenAI 开发的生成式预训练Transformer模型系列。其核心思想是：

- 生成式：能够生成连贯的文本

- 预训练：在大量无标签文本上进行自监督学习

- Transformer：基于 Transformer 解码器架构

## 环境配置

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10    | 2.7.0       | 0.5.1           |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [ ]:
# !pip install mindspore==2.7.0 mindnlp==0.5.1

其他场景可参考[MindSpore安装指南](https://www.mindspore.cn/install)与[MindSpore NLP安装指南](https://github.com/mindspore-lab/mindnlp?tab=readme-ov-file#installation)进行环境搭建。

## 安装依赖

In [ ]:
!pip install jieba

In [ ]:
import os
import mindnlp
import mindspore
from mindspore.dataset import text, GeneratorDataset, transforms
from mindspore import nn

from datasets import load_dataset

from transformers import Trainer

/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress 

In [ ]:
mindspore.set_context(pynative_synchronize=True) #开启同步设置，方便后续定位

[WARNING] ME(13257:281473667298880,MainProcess):2025-11-25-14:35:30.889.000 [mindspore/context.py:1412] For 'context.set_context', the parameter 'pynative_synchronize' will be deprecated and removed in a future version. Please use the api mindspore.runtime.launch_blocking() instead.


## 数据加载与预处理

### 数据集加载

In [5]:
# 在加载时直接采样10%的数据
imdb_ds = load_dataset('imdb', split=['train[:10%]', 'test[:10%]'])
imdb_train = imdb_ds[0]  # 10%的训练数据
imdb_test = imdb_ds[1]   # 10%的测试数据

print(f"采样后训练集大小: {len(imdb_train)}")
print(f"采样后测试集大小: {len(imdb_test)}")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 153439.56 examples/s]

采样后训练集大小: 2500
采样后测试集大小: 2500


### 数据预处理

In [6]:
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding

def process_dataset(dataset, tokenizer, max_seq_len=512, batch_size=4, shuffle=False):
    
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_seq_len,
            return_tensors=None
        )
    
    # 如果 shuffle 为 True，先打乱数据集
    if shuffle:
        dataset = dataset.shuffle(seed=42)
    
    # 应用 tokenize 函数
    dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=['text']
    )
    
    # 重命名标签列
    if 'label' in dataset.column_names:
        dataset = dataset.rename_column('label', 'labels')
    
    # 设置格式
    dataset.set_format(type='torch')
    
    # 使用 DataCollatorWithPadding 处理所有 padding
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding='longest'  # 动态 padding
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=data_collator,
        drop_last=False
    )
    
    return dataloader

In [7]:
from transformers import OpenAIGPTTokenizer
# tokenizer
gpt_tokenizer = OpenAIGPTTokenizer.from_pretrained('openai-gpt')

# add sepcial token: <PAD>
special_tokens_dict = {
    "bos_token": "<bos>",
    "eos_token": "<eos>",
    "pad_token": "<pad>",
}
num_added_toks = gpt_tokenizer.add_special_tokens(special_tokens_dict)

ftfy or spacy is not installed using BERT BasicTokenizer instead of SpaCy & ftfy.


In [8]:
# split train dataset into train and valid datasets
dataset_split = imdb_train.train_test_split(test_size=0.3, seed=42)
imdb_train = dataset_split['train']
imdb_val = dataset_split['test']

print(f"训练集大小: {len(imdb_train)}")
print(f"验证集大小: {len(imdb_val)}")

训练集大小: 1750
验证集大小: 750


In [9]:
dataset_train = process_dataset(imdb_train, gpt_tokenizer, shuffle=True)
dataset_val = process_dataset(imdb_val, gpt_tokenizer)
dataset_test = process_dataset(imdb_test, gpt_tokenizer)

Map: 100%|██████████| 2500/2500 [00:21<00:00, 113.66 examples/s]


### 查看数据信息

In [10]:
# 检查数据
print("检查训练数据:")
for i, batch in enumerate(dataset_train):
    print(f"Batch {i}:")
    print(f"  Input IDs shape: {batch['input_ids'].shape}")
    print(f"  Attention mask shape: {batch['attention_mask'].shape}")
    print(f"  Labels: {batch['labels']}")
    if i >= 2:  # 只看前3个batch
        break

检查训练数据:
[MS_ALLOC_CONF]Runtime config:  enable_vmm:True  vmm_align_size:2MB
Batch 0:
  Input IDs shape: mindtorch.Size([4, 268])
  Attention mask shape: mindtorch.Size([4, 268])
  Labels: [0 0 0 0]
Batch 1:
  Input IDs shape: mindtorch.Size([4, 512])
  Attention mask shape: mindtorch.Size([4, 512])
  Labels: [0 0 0 0]
Batch 2:
  Input IDs shape: mindtorch.Size([4, 512])
  Attention mask shape: mindtorch.Size([4, 512])
  Labels: [0 0 0 0]


In [ ]:
# 现有的 process_dataset 函数返回 DataLoader，可以这样提取 Dataset
def extract_dataset_from_dataloader(dataloader):
    """从 DataLoader 中提取原始的 Dataset"""
    return dataloader.dataset

# 使用示例
dataset_train = extract_dataset_from_dataloader(dataset_train)
dataset_val = extract_dataset_from_dataloader(dataset_val)
dataset_test = extract_dataset_from_dataloader(dataset_test)

## 加载评估指标

In [ ]:
!pip install scikit-learn  # 安装依赖

Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 133.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [13]:
import evaluate
import numpy as np
from transformers import EvalPrediction

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

## 模型加载

In [ ]:
from transformers import OpenAIGPTForSequenceClassification, TrainingArguments


model = OpenAIGPTForSequenceClassification.from_pretrained('openai-gpt', num_labels=2)
model.config.pad_token_id = gpt_tokenizer.pad_token_id
model.resize_token_embeddings(model.config.vocab_size + 3)

In [ ]:
training_args = TrainingArguments(
    "./output/gpt",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=200,
    eval_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(model=model, train_dataset=dataset_train,
                  eval_dataset=dataset_val, compute_metrics=compute_metrics,
                  args=training_args)

Some weights of OpenAIGPTForSequenceClassification were not initialized from the model checkpoint at openai-gpt and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Detected kernel version 4.19.90, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## 模型训练

In [ ]:
trainer.train()

## 模型评估

In [ ]:
model.npu()  #模型移动到npu侧
device = model.device  # 获取模型所在的设备
device

device(type=npu, index=0)

In [20]:
from tqdm import tqdm
import torch
import numpy as np

def compute_accuracy(logits, labels):    
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

def evaluate_fn(model, test_dataset):
    # 创建 DataLoader（因为 test_dataset 应该是 Dataset 对象）
    from torch.utils.data import DataLoader
    from transformers import DataCollatorWithPadding
    
    data_collator = DataCollatorWithPadding(tokenizer=gpt_tokenizer)
    dataloader = DataLoader(
        test_dataset,
        batch_size=1,
        collate_fn=data_collator,
        shuffle=False
    )
    
    total = len(test_dataset)
    epoch_acc = 0
    step_total = 0
    
    model.eval()  # 替代 model.set_train(False)

    with torch.no_grad():  # 禁用梯度计算
        with tqdm(total=total) as progress_bar:
            for batch in dataloader:
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop('labels')
                logits = model(**batch).logits

                acc = compute_accuracy(logits, labels)['accuracy']
                epoch_acc += acc
                step_total += 1
                current_acc = epoch_acc / step_total
                # 更新进度条
                progress_bar.set_postfix({'current_accuracy': f'{current_acc:.4f}'})
                progress_bar.update(1)

    final_accuracy = epoch_acc / step_total
    print(f"最终准确率: {final_accuracy:.4f}")
    return final_accuracy

In [23]:
acc = evaluate_fn(model, dataset_val)
print(f"Accuracy: {acc}")

100%|██████████| 750/750 [01:30<00:00,  8.31it/s, current_accuracy=1.0000]

最终准确率: 1.0000
Accuracy: 1.0
